# Docker 第4周：进阶与实战

> **学习目标**：掌握 CI/CD 集成、安全加固、监控日志、排障技巧

---

## Docker 与 CI/CD

最典型的场景：代码 push 到 GitHub → GitHub Actions 自动构建镜像 → 推送到 registry → 服务器拉取部署。

### GitHub Actions 示例

```yaml
# .github/workflows/docker-build.yml
name: Build and Push Docker Image

on:
  push:
    branches: [main]
    tags: ['v*']

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Login to Docker Hub
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKER_USERNAME }}
          password: ${{ secrets.DOCKER_TOKEN }}

      - name: Build and push
        uses: docker/build-push-action@v5
        with:
          push: true
          tags: |
            user/app:latest
            user/app:${{ github.sha }}
          cache-from: type=gha
          cache-to: type=gha,mode=max
```

关键点：
- 用 GitHub Actions cache 加速构建（`cache-from` / `cache-to`）
- 同时打 `latest` 和 `commit-sha` 两个 tag
- Token 存在 GitHub Secrets 里，不暴露在代码中

---

## 监控：知道容器在干什么

### `docker stats`：实时资源监控

In [ ]:
# 启动几个容器制造负载，然后监控
! docker run -d --name stress-test python:3.12-slim \
  python -c "import time; [time.sleep(0.1) for _ in range(10000)]" 2>/dev/null || echo "已启动"

# 查看资源使用（只跑一次，不加 --stream）
! docker stats --no-stream

In [ ]:
# 清理
! docker rm -f stress-test 2>/dev/null
print("已清理")

### 日志驱动

Docker 支持多种日志驱动：

| 驱动 | 说明 | 场景 |
|------|------|------|
| `json-file` | Docker 默认，JSON 格式存本地 | 开发调试 |
| `syslog` | 发送到系统 syslog | 传统运维 |
| `journald` | 发送到 systemd journal | Linux 系统 |
| `fluentd` | 发送到 Fluentd 收集器 | ELK/EFK 日志栈 |
| `gelf` | 发送到 Graylog | 集中式日志 |
| `awslogs` / `gcplogs` | 云平台日志服务 | 云上部署 |

配置方式（daemon 级别或容器级别）：

```bash
# 容器级别
docker run --log-driver=fluentd --log-opt fluentd-address=localhost:24224 ...

# compose 中
logging:
  driver: "json-file"
  options:
    max-size: "10m"
    max-file: "3"
```

---

## 安全清单

Docker 安全不是"配置一个选项"，而是一组习惯：

### 1. 最小权限原则

```dockerfile
# ✓ 好
FROM python:3.12-slim
RUN useradd --create-home --shell /bin/bash appuser
USER appuser
CMD ["python", "app.py"]

# ✗ 坏：以 root 运行
FROM python:3.12-slim
CMD ["python", "app.py"]
```

### 2. 镜像扫描

```bash
# Docker Scout（Docker 内置）
docker scout quickview my-image

# Trivy（第三方开源）
trivy image my-image
```

### 3. 只读根文件系统

```bash
docker run --read-only --tmpfs /tmp my-app
```
应用只能写 `/tmp`，其他地方只读。即使被攻破，攻击者也无法修改系统文件。

### 4. 限制能力

```bash
docker run --cap-drop=ALL --cap-add=NET_BIND_SERVICE my-app
```
`--cap-drop=ALL` 去掉所有 Linux capabilities，然后按需加回。一个 Web 应用基本只需要 `NET_BIND_SERVICE`（绑定 80 端口）。

### 5. 密钥管理

**不要**把密钥写进 Dockerfile 或环境变量！

```dockerfile
# ✗ 绝对禁止：密码写在 Dockerfile 里（会被提交到 Git，且 docker history 可见！）
ENV DATABASE_PASSWORD=mysecretpassword

# ✗ 也不太安全：构建时传入（docker history 仍然可见）
# docker build --build-arg DB_PASS=xxx .
ARG DB_PASS
ENV DATABASE_PASSWORD=$DB_PASS
```

推荐方案：
- **运行时注入**：`docker run -e DB_PASS=xxx` 或 Compose 的 `env_file`
- **Secrets 管理**：Docker Swarm Secrets / K8s Secrets / HashiCorp Vault
- **BuildKit secrets**（构建时需要用密钥，但不在镜像中留存）：

```dockerfile
# syntax=docker/dockerfile:1
RUN --mount=type=secret,id=mysecret cat /run/secrets/mysecret
```

```bash
docker build --secret id=mysecret,src=./secret.txt -t my-app .
```

---

## 排障手册

### 场景 1：容器起不来

```bash
# 第一步：看日志
docker logs <container>

# 第二步：看详细信息（Events 部分最有用）
docker inspect <container> | jq '.[0].State'

# 常见原因：
# - 镜像不存在 / 拉取失败
# - 端口被占用
# - 资源不够（内存/CPU）
# - CMD/ENTRYPOINT 写错了
# - 依赖服务没就绪
```

### 场景 2：服务访问不通

```bash
# 检查端口映射
docker port <container>

# 进入容器 curl 自己
docker exec <container> curl localhost:5000

# 检查网络
docker network inspect <network>

# 常见原因：
# - 应用监听了 127.0.0.1 而不是 0.0.0.0
# - 端口映射写反了
# - 防火墙
# - NetworkPolicy（K8s）/ iptables
```

In [ ]:
# 故意制造一个错误并排查
! mkdir -p /tmp/docker-debug

%%writefile /tmp/docker-debug/bug.py
# 这个脚本监听 127.0.0.1，外部无法访问！
from http.server import HTTPServer, SimpleHTTPRequestHandler
import sys

port = int(sys.argv[1]) if len(sys.argv) > 1 else 8000
# BUG! 应该是 0.0.0.0
server = HTTPServer(("127.0.0.1", port), SimpleHTTPRequestHandler)
print(f"监听 127.0.0.1:{port}（外部无法访问！）")
server.serve_forever()

%%writefile /tmp/docker-debug/Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY bug.py .
EXPOSE 8000
CMD ["python", "bug.py"]

In [ ]:
! docker build -t debug-test /tmp/docker-debug -q
! docker run -d --name debug-test -p 8000:8000 debug-test
! sleep 2

# 容器在跑，但访问不通！
! curl -s -o /dev/null -w "HTTP %{http_code}" http://localhost:8000 || echo " → 连接失败！"

# 排障：进入容器内部 curl
! docker exec debug-test apt-get update -qq && docker exec debug-test apt-get install -y -qq curl 2>/dev/null
! docker exec debug-test curl -s -o /dev/null -w "容器内部访问: HTTP %{http_code}\n" http://localhost:8000
! docker exec debug-test curl -s -o /dev/null -w "容器内部访问 0.0.0.0: HTTP %{http_code}\n" http://0.0.0.0:8000 || echo "0.0.0.0 也不通"

# 结论：应用只监听了 127.0.0.1，Docker 端口映射流量走的是 0.0.0.0，所以不通
! docker rm -f debug-test

### 场景 3：磁盘满了

In [ ]:
# 查看 Docker 占用
! docker system df

# 详细信息
! docker system df -v | head -30

# 清理
# ! docker system prune -a --volumes  # 慎用！会删所有未使用的东西

print("\n清理建议：")
print("  docker image prune    # 清理悬空镜像")
print("  docker container prune # 清理停止的容器")
print("  docker volume prune    # 清理未使用的卷")
print("  docker builder prune   # 清理构建缓存")
print("  docker system prune    # 一键清理（慎用 -a）")

---

## 常用工具推荐

| 工具 | 用途 | 命令 |
|------|------|------|
| **dive** | 分析镜像分层，看每层增加了什么 | `dive my-image` |
| **ctop** | 容器实时监控（类 htop） | `ctop` |
| **lazydocker** | 终端 UI 管理 Docker | `lazydocker` |
| **hadolint** | Dockerfile 静态检查 | `hadolint Dockerfile` |
| **trivy** | 镜像漏洞扫描 | `trivy image my-image` |
| **dockle** | 镜像最佳实践检查 | `dockle my-image` |
| **docker-compose-viz** | compose 可视化 | 生成服务拓扑图 |

---

## 生产级 Dockerfile 模板

以下是经过优化的 Python 应用 Dockerfile 模板，整合了前三周学到的所有最佳实践：

In [ ]:
%%writefile /tmp/docker-debug/Dockerfile.prod
# ===========================
# 生产级 Python Dockerfile 模板
# ===========================

# ---- Stage 1: 构建依赖 ----
FROM python:3.12-slim AS builder

# 只安装编译依赖
RUN apt-get update && \
    apt-get install -y --no-install-recommends gcc libpq-dev && \
    rm -rf /var/lib/apt/lists/*

WORKDIR /app
COPY requirements.txt .
# --user 安装到 /root/.local，方便后续 COPY
RUN pip install --user --no-cache-dir -r requirements.txt

# ---- Stage 2: 运行 ----
FROM python:3.12-slim

# 创建非 root 用户
RUN groupadd -r appuser && useradd -r -g appuser -s /bin/bash appuser

WORKDIR /app

# 从构建阶段复制已安装的包
COPY --from=builder /root/.local /home/appuser/.local
ENV PATH=/home/appuser/.local/bin:$PATH

# 复制应用代码
COPY --chown=appuser:appuser app.py .

# 切换到非 root
USER appuser

# 健康检查
HEALTHCHECK --interval=30s --timeout=3s --retries=3 \
  CMD python -c "from urllib.request import urlopen; urlopen('http://localhost:5000/health')" || exit 1

EXPOSE 5000
CMD ["python", "app.py"]

print("✓ 生产级 Dockerfile 已生成")

---

## 🎯 第4周总结

| 主题 | 核心要点 |
|------|----------|
| **CI/CD** | GitHub Actions 自动构建+推送，利用缓存加速 |
| **监控** | `docker stats` / `docker events` / cAdvisor / 日志驱动 |
| **安全** | 非 root / 只读根文件系统 / cap-drop / 密钥不入镜像 |
| **排障** | 日志 → inspect → 进容器 curl → 查网络 → 查资源 |
| **工具** | dive / hadolint / trivy / lazydocker |

### 四周学习回顾

```
Week 1: docker run / ps / stop / exec / logs — 容器生命周期
    ↓
Week 2: Dockerfile / 构建优化 / 多阶段构建 / .dockerignore — 造镜像
    ↓
Week 3: Compose / 多服务编排 / 多环境 / 网络与卷 — 管多个服务
    ↓
Week 4: CI/CD / 安全 / 监控 / 排障 — 生产就绪
```

---

## 🧪 最终项目：把自己的 Python 项目全部容器化

选择你之前写过的 Python 项目（Todo / Flask API / 爬虫），完成以下任务：

1. **编写生产级 Dockerfile**（非 root / 多阶段 / 健康检查）
2. **编写 Compose 文件**（如果涉及多服务）
3. **编写 .dockerignore**
4. **本地构建并测试**
5. **（可选）推送至 Docker Hub**
6. **写一份 DOCKER.md 文档**，记录：
   - 如何构建镜像
   - 如何运行容器
   - 常用的调试命令
   - 踩过的坑和解决方案

In [ ]:
# 你的最终项目代码写在这里
pass